# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'archive/prompt_discovery/prompt2'))


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

# ============================================================
# Configuration
# ============================================================

MIN_LAYER = 17
TOP_K = 20
CORE_EFFECT_MASS = 0.75

EMOTION_COLUMNS = [
    "logit_anger",
    "logit_disgust",
    "logit_fear",
    "logit_joy",
    "logit_sadness",
    "logit_surprise",
]

# ============================================================
# Load data
# ============================================================

original_features = pd.read_csv(
    "../data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

prompt2 = pd.read_csv(
    "data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions_prompt2.csv"
)

prompt3 = pd.read_csv(
    "prompt3/data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions_prompt3.csv"
)

prompt4 = pd.read_csv(
    "prompt4/data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions_prompt4.csv"
)

feature_dfs = {
    "original": original_features,
    "prompt2": prompt2,
    "prompt3": prompt3,
    "prompt4": prompt4,
}

# ============================================================
# Validate and preprocess
# ============================================================

required_columns = {
    "layer_idx",
    "feature_idx",
    *EMOTION_COLUMNS,
}

for name, df in feature_dfs.items():
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing_columns)}"
        )

    df = df.copy()

    # Keep layers 17 and later
    df = df[df["layer_idx"] >= MIN_LAYER]

    # Convert logit columns to numeric
    for column in EMOTION_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0.0)

    # Aggregate duplicate feature rows if necessary
    df = (
        df.groupby(
            ["layer_idx", "feature_idx"],
            as_index=False,
        )[EMOTION_COLUMNS]
        .mean()
    )

    feature_dfs[name] = df


# ============================================================
# Helper functions
# ============================================================

def get_effect_dictionary(df, emotion_column):
    """
    Return:
        {(layer_idx, feature_idx): absolute_logit_effect}
    """
    return {
        (row.layer_idx, row.feature_idx): abs(
            getattr(row, emotion_column)
        )
        for row in df.itertuples(index=False)
    }


def get_top_k_features(effect_dict, top_k):
    """
    Return the feature IDs with the largest absolute effects.
    """
    sorted_features = sorted(
        effect_dict.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        feature
        for feature, effect in sorted_features[:top_k]
        if effect > 0
    }


def get_core_mass_features(effect_dict, mass_threshold=0.80):
    """
    Return the smallest feature set whose cumulative absolute effect
    reaches the specified fraction of total effect mass.
    """
    sorted_features = sorted(
        effect_dict.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    total_effect = sum(effect for _, effect in sorted_features)

    if total_effect == 0:
        return set()

    selected_features = set()
    cumulative_effect = 0.0

    for feature, effect in sorted_features:
        if effect <= 0:
            continue

        selected_features.add(feature)
        cumulative_effect += effect

        if cumulative_effect / total_effect >= mass_threshold:
            break

    return selected_features


def jaccard_similarity(set1, set2):
    union = set1 | set2

    if not union:
        return np.nan

    return len(set1 & set2) / len(union)


def overlap_coefficient(set1, set2):
    """
    Intersection divided by the size of the smaller set.
    This measures how much the smaller core is contained in the larger core.
    """
    denominator = min(len(set1), len(set2))

    if denominator == 0:
        return np.nan

    return len(set1 & set2) / denominator


def weighted_jaccard(effect_dict1, effect_dict2):
    """
    Weighted Jaccard:
        sum(min(effect_1, effect_2)) / sum(max(effect_1, effect_2))

    Strong features contribute more than weak features.
    """
    all_features = set(effect_dict1) | set(effect_dict2)

    if not all_features:
        return np.nan

    numerator = sum(
        min(
            effect_dict1.get(feature, 0.0),
            effect_dict2.get(feature, 0.0),
        )
        for feature in all_features
    )

    denominator = sum(
        max(
            effect_dict1.get(feature, 0.0),
            effect_dict2.get(feature, 0.0),
        )
        for feature in all_features
    )

    if denominator == 0:
        return np.nan

    return numerator / denominator


# ============================================================
# Compute emotion-specific overlaps
# ============================================================

results = []

for emotion_column in EMOTION_COLUMNS:
    emotion = emotion_column.replace("logit_", "")

    effect_dicts = {
        prompt_name: get_effect_dictionary(df, emotion_column)
        for prompt_name, df in feature_dfs.items()
    }

    top_k_sets = {
        prompt_name: get_top_k_features(effect_dict, TOP_K)
        for prompt_name, effect_dict in effect_dicts.items()
    }

    core_mass_sets = {
        prompt_name: get_core_mass_features(
            effect_dict,
            CORE_EFFECT_MASS,
        )
        for prompt_name, effect_dict in effect_dicts.items()
    }

    for prompt1, prompt2_name in combinations(feature_dfs.keys(), 2):
        top_set1 = top_k_sets[prompt1]
        top_set2 = top_k_sets[prompt2_name]

        core_set1 = core_mass_sets[prompt1]
        core_set2 = core_mass_sets[prompt2_name]

        results.append(
            {
                "emotion": emotion,
                "prompt_1": prompt1,
                "prompt_2": prompt2_name,

                # Top-k overlap
                "top_k": TOP_K,
                "top_k_shared": len(top_set1 & top_set2),
                "top_k_jaccard": jaccard_similarity(
                    top_set1,
                    top_set2,
                ),
                "top_k_overlap_coefficient": overlap_coefficient(
                    top_set1,
                    top_set2,
                ),

                # Core effect-mass overlap
                "core_mass_threshold": CORE_EFFECT_MASS,
                "num_core_features_1": len(core_set1),
                "num_core_features_2": len(core_set2),
                "core_shared": len(core_set1 & core_set2),
                "core_jaccard": jaccard_similarity(
                    core_set1,
                    core_set2,
                ),
                "core_overlap_coefficient": overlap_coefficient(
                    core_set1,
                    core_set2,
                ),

                # Effect-weighted overlap
                "weighted_jaccard": weighted_jaccard(
                    effect_dicts[prompt1],
                    effect_dicts[prompt2_name],
                ),
            }
        )

overlap_df = pd.DataFrame(results)

print("\nEmotion-specific core feature overlap:")
print(
    overlap_df.round(4).to_string(index=False)
)


# ============================================================
# Average across prompt pairs for each emotion
# ============================================================

emotion_summary = (
    overlap_df.groupby("emotion", as_index=False)
    .agg(
        mean_top_k_shared=("top_k_shared", "mean"),
        mean_top_k_jaccard=("top_k_jaccard", "mean"),
        mean_top_k_overlap_coefficient=(
            "top_k_overlap_coefficient",
            "mean",
        ),
        mean_core_shared=("core_shared", "mean"),
        mean_core_jaccard=("core_jaccard", "mean"),
        mean_core_overlap_coefficient=(
            "core_overlap_coefficient",
            "mean",
        ),
        mean_weighted_jaccard=("weighted_jaccard", "mean"),
    )
)

print("\nAverage overlap by emotion:")
print(
    emotion_summary.round(4).to_string(index=False)
)


# ============================================================
# Features shared across all three prompts
# ============================================================

shared_core_results = []

for emotion_column in EMOTION_COLUMNS:
    emotion = emotion_column.replace("logit_", "")

    effect_dicts = {
        prompt_name: get_effect_dictionary(df, emotion_column)
        for prompt_name, df in feature_dfs.items()
    }

    top_k_sets = {
        prompt_name: get_top_k_features(effect_dict, TOP_K)
        for prompt_name, effect_dict in effect_dicts.items()
    }

    core_mass_sets = {
        prompt_name: get_core_mass_features(
            effect_dict,
            CORE_EFFECT_MASS,
        )
        for prompt_name, effect_dict in effect_dicts.items()
    }

    shared_top_k = set.intersection(*top_k_sets.values())
    shared_core_mass = set.intersection(*core_mass_sets.values())

    shared_core_results.append(
        {
            "emotion": emotion,
            "shared_top_k_all_prompts": len(shared_top_k),
            "shared_top_k_fraction": (
                len(shared_top_k) / TOP_K
                if TOP_K > 0
                else np.nan
            ),
            "shared_core_all_prompts": len(shared_core_mass),
            "smallest_core_size": min(
                len(feature_set)
                for feature_set in core_mass_sets.values()
            ),
            "shared_core_fraction_of_smallest": (
                len(shared_core_mass)
                / min(
                    len(feature_set)
                    for feature_set in core_mass_sets.values()
                )
                if min(
                    len(feature_set)
                    for feature_set in core_mass_sets.values()
                ) > 0
                else np.nan
            ),
        }
    )

shared_core_df = pd.DataFrame(shared_core_results)

print("\nFeatures shared across all three prompts:")
print(
    shared_core_df.round(4).to_string(index=False)
)


# ============================================================
# Save outputs
# ============================================================

overlap_df.to_csv(
    "emotion_core_feature_pairwise_overlap.csv",
    index=False,
)

emotion_summary.to_csv(
    "emotion_core_feature_overlap_summary.csv",
    index=False,
)

shared_core_df.to_csv(
    "emotion_core_features_shared_all_prompts.csv",
    index=False,
)

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

# ============================================================
# Configuration
# ============================================================

MIN_LAYER = 22
TOP_K = 10
CORE_EFFECT_MASS = 1.0

EMOTION_COLUMNS = [
    "logit_anger",
    "logit_disgust",
    "logit_fear",
    "logit_joy",
    "logit_sadness",
    "logit_surprise",
]

# ============================================================
# Load data
# ============================================================

original_features = pd.read_csv("../data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions.csv")
prompt2 = pd.read_csv("data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions_prompt2.csv")
prompt3 = pd.read_csv("prompt3/data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions_prompt3.csv")
prompt4 = pd.read_csv("prompt4/data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions_prompt4.csv")

feature_dfs = {
    "original": original_features,
    "prompt2": prompt2,
    "prompt3": prompt3,
    "prompt4": prompt4,
}

# ============================================================
# Validate and preprocess
# ============================================================

required_columns = {
    "layer_idx",
    "feature_idx",
    *EMOTION_COLUMNS,
}

for name, df in feature_dfs.items():
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing_columns)}"
        )

    df = df.copy()

    # Keep layers 17 and later
    df = df[df["layer_idx"] >= MIN_LAYER]

    # Convert logit columns to numeric
    for column in EMOTION_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0.0)

    # Aggregate duplicate feature rows if necessary
    df = (
        df.groupby(
            ["layer_idx", "feature_idx"],
            as_index=False,
        )[EMOTION_COLUMNS]
        .mean()
    )

    feature_dfs[name] = df


# ============================================================
# Helper functions
# ============================================================

def get_effect_dictionary(df, emotion_column):
    """
    Return:
        {(layer_idx, feature_idx): absolute_logit_effect}
    """
    return {
        (row.layer_idx, row.feature_idx): abs(
            getattr(row, emotion_column)
        )
        for row in df.itertuples(index=False)
    }


def get_top_k_features(effect_dict, top_k):
    """
    Return the feature IDs with the largest absolute effects.
    """
    sorted_features = sorted(
        effect_dict.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        feature
        for feature, effect in sorted_features[:top_k]
        if effect > 0
    }


def get_core_mass_features(effect_dict, mass_threshold=0.80):
    """
    Return the smallest feature set whose cumulative absolute effect
    reaches the specified fraction of total effect mass.
    """
    sorted_features = sorted(
        effect_dict.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    total_effect = sum(effect for _, effect in sorted_features)

    if total_effect == 0:
        return set()

    selected_features = set()
    cumulative_effect = 0.0

    for feature, effect in sorted_features:
        if effect <= 0:
            continue

        selected_features.add(feature)
        cumulative_effect += effect

        if cumulative_effect / total_effect >= mass_threshold:
            break

    return selected_features


def jaccard_similarity(set1, set2):
    union = set1 | set2

    if not union:
        return np.nan

    return len(set1 & set2) / len(union)


def overlap_coefficient(set1, set2):
    """
    Intersection divided by the size of the smaller set.
    This measures how much the smaller core is contained in the larger core.
    """
    denominator = min(len(set1), len(set2))

    if denominator == 0:
        return np.nan

    return len(set1 & set2) / denominator


def weighted_jaccard(effect_dict1, effect_dict2):
    """
    Weighted Jaccard:
        sum(min(effect_1, effect_2)) / sum(max(effect_1, effect_2))

    Strong features contribute more than weak features.
    """
    all_features = set(effect_dict1) | set(effect_dict2)

    if not all_features:
        return np.nan

    numerator = sum(
        min(
            effect_dict1.get(feature, 0.0),
            effect_dict2.get(feature, 0.0),
        )
        for feature in all_features
    )

    denominator = sum(
        max(
            effect_dict1.get(feature, 0.0),
            effect_dict2.get(feature, 0.0),
        )
        for feature in all_features
    )

    if denominator == 0:
        return np.nan

    return numerator / denominator


# ============================================================
# Compute emotion-specific overlaps
# ============================================================

results = []

for emotion_column in EMOTION_COLUMNS:
    emotion = emotion_column.replace("logit_", "")

    effect_dicts = {
        prompt_name: get_effect_dictionary(df, emotion_column)
        for prompt_name, df in feature_dfs.items()
    }

    top_k_sets = {
        prompt_name: get_top_k_features(effect_dict, TOP_K)
        for prompt_name, effect_dict in effect_dicts.items()
    }

    core_mass_sets = {
        prompt_name: get_core_mass_features(
            effect_dict,
            CORE_EFFECT_MASS,
        )
        for prompt_name, effect_dict in effect_dicts.items()
    }

    for prompt1, prompt2_name in combinations(feature_dfs.keys(), 2):
        top_set1 = top_k_sets[prompt1]
        top_set2 = top_k_sets[prompt2_name]

        core_set1 = core_mass_sets[prompt1]
        core_set2 = core_mass_sets[prompt2_name]

        results.append(
            {
                "emotion": emotion,
                "prompt_1": prompt1,
                "prompt_2": prompt2_name,

                # Top-k overlap
                "top_k": TOP_K,
                "top_k_shared": len(top_set1 & top_set2),
                "top_k_jaccard": jaccard_similarity(
                    top_set1,
                    top_set2,
                ),
                "top_k_overlap_coefficient": overlap_coefficient(
                    top_set1,
                    top_set2,
                ),

                # Core effect-mass overlap
                "core_mass_threshold": CORE_EFFECT_MASS,
                "num_core_features_1": len(core_set1),
                "num_core_features_2": len(core_set2),
                "core_shared": len(core_set1 & core_set2),
                "core_jaccard": jaccard_similarity(
                    core_set1,
                    core_set2,
                ),
                "core_overlap_coefficient": overlap_coefficient(
                    core_set1,
                    core_set2,
                ),

                # Effect-weighted overlap
                "weighted_jaccard": weighted_jaccard(
                    effect_dicts[prompt1],
                    effect_dicts[prompt2_name],
                ),
            }
        )

overlap_df = pd.DataFrame(results)

print("\nEmotion-specific core feature overlap:")
print(
    overlap_df.round(4).to_string(index=False)
)


# ============================================================
# Average across prompt pairs for each emotion
# ============================================================

emotion_summary = (
    overlap_df.groupby("emotion", as_index=False)
    .agg(
        mean_top_k_shared=("top_k_shared", "mean"),
        mean_top_k_jaccard=("top_k_jaccard", "mean"),
        mean_top_k_overlap_coefficient=(
            "top_k_overlap_coefficient",
            "mean",
        ),
        mean_core_shared=("core_shared", "mean"),
        mean_core_jaccard=("core_jaccard", "mean"),
        mean_core_overlap_coefficient=(
            "core_overlap_coefficient",
            "mean",
        ),
        mean_weighted_jaccard=("weighted_jaccard", "mean"),
    )
)

print("\nAverage overlap by emotion:")
print(
    emotion_summary.round(4).to_string(index=False)
)


# ============================================================
# Features shared across all three prompts
# ============================================================

shared_core_results = []

for emotion_column in EMOTION_COLUMNS:
    emotion = emotion_column.replace("logit_", "")

    effect_dicts = {
        prompt_name: get_effect_dictionary(df, emotion_column)
        for prompt_name, df in feature_dfs.items()
    }

    top_k_sets = {
        prompt_name: get_top_k_features(effect_dict, TOP_K)
        for prompt_name, effect_dict in effect_dicts.items()
    }

    core_mass_sets = {
        prompt_name: get_core_mass_features(
            effect_dict,
            CORE_EFFECT_MASS,
        )
        for prompt_name, effect_dict in effect_dicts.items()
    }

    shared_top_k = set.intersection(*top_k_sets.values())
    shared_core_mass = set.intersection(*core_mass_sets.values())

    shared_core_results.append(
        {
            "emotion": emotion,
            "shared_top_k_all_prompts": len(shared_top_k),
            "shared_top_k_fraction": (
                len(shared_top_k) / TOP_K
                if TOP_K > 0
                else np.nan
            ),
            "shared_core_all_prompts": len(shared_core_mass),
            "smallest_core_size": min(
                len(feature_set)
                for feature_set in core_mass_sets.values()
            ),
            "shared_core_fraction_of_smallest": (
                len(shared_core_mass)
                / min(
                    len(feature_set)
                    for feature_set in core_mass_sets.values()
                )
                if min(
                    len(feature_set)
                    for feature_set in core_mass_sets.values()
                ) > 0
                else np.nan
            ),
        }
    )

shared_core_df = pd.DataFrame(shared_core_results)

print("\nFeatures shared across all three prompts:")
print(
    shared_core_df.round(4).to_string(index=False)
)


# ============================================================
# Save outputs
# ============================================================

overlap_df.to_csv(
    "emotion_core_feature_pairwise_overlap.csv",
    index=False,
)

emotion_summary.to_csv(
    "emotion_core_feature_overlap_summary.csv",
    index=False,
)

shared_core_df.to_csv(
    "emotion_core_features_shared_all_prompts.csv",
    index=False,
)

In [ ]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

EMOTION_COLUMNS = [
    "logit_anger",
    "logit_disgust",
    "logit_fear",
    "logit_joy",
    "logit_sadness",
    "logit_surprise",
]

EMOTION_ORDER = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

# Each model can use its own layer cutoff and core-mass threshold.
MODEL_CONFIGS = {
    "Gemma-2-2B": {
        "min_layer": 17,
        "core_effect_mass": 0.75,
        "prompt_paths": {
            "original": (
                "../data/google-gemma-2-2b-feature-act/"
                "auto_selected_unique_features_with_affected_emotions.csv"
            ),
            "prompt2": (
                "data/google-gemma-2-2b-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt2.csv"
            ),
            "prompt3": (
                "prompt3/data/google-gemma-2-2b-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt3.csv"
            ),
            "prompt4": (
                "prompt4/data/google-gemma-2-2b-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt4.csv"
            ),
        },
    },
    "Llama-3.1-8B": {
        "min_layer": 22,
        "core_effect_mass": 1.0,
        "prompt_paths": {
            "original": (
                "../data/meta-llama-Llama-3.1-8B-feature-act/"
                "auto_selected_unique_features_with_affected_emotions.csv"
            ),
            "prompt2": (
                "data/meta-llama-Llama-3.1-8B-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt2.csv"
            ),
            "prompt3": (
                "prompt3/data/meta-llama-Llama-3.1-8B-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt3.csv"
            ),
            "prompt4": (
                "prompt4/data/meta-llama-Llama-3.1-8B-feature-act/"
                "auto_selected_unique_features_with_affected_emotions_prompt4.csv"
            ),
        },
    },
}

OUTPUT_DIR = Path("core_overlap_tables")


# ============================================================
# Helpers
# ============================================================

def load_and_preprocess(path: str, min_layer: int) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {"layer_idx", "feature_idx", *EMOTION_COLUMNS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")

    df = df[df["layer_idx"] >= min_layer].copy()

    for column in EMOTION_COLUMNS:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        ).fillna(0.0)

    df = (
        df.groupby(
            ["layer_idx", "feature_idx"],
            as_index=False,
        )[EMOTION_COLUMNS]
        .mean()
    )

    return df


def get_effect_dictionary(
    df: pd.DataFrame,
    emotion_column: str,
) -> Dict[tuple, float]:
    return {
        (int(row.layer_idx), int(row.feature_idx)): abs(
            float(getattr(row, emotion_column))
        )
        for row in df.itertuples(index=False)
    }


def get_core_mass_features(
    effect_dict: Dict[tuple, float],
    mass_threshold: float,
) -> set:
    """
    Return the smallest feature set whose cumulative absolute effect
    reaches the requested fraction of total effect mass.
    """
    sorted_features = sorted(
        effect_dict.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    total_effect = sum(effect for _, effect in sorted_features)

    if total_effect <= 0:
        return set()

    selected = set()
    cumulative_effect = 0.0

    for feature, effect in sorted_features:
        if effect <= 0:
            continue

        selected.add(feature)
        cumulative_effect += effect

        if cumulative_effect / total_effect >= mass_threshold:
            break

    return selected


def overlap_coefficient(set1: set, set2: set) -> float:
    denominator = min(len(set1), len(set2))

    if denominator == 0:
        return np.nan

    return len(set1 & set2) / denominator


# ============================================================
# Compute model-level results
# ============================================================

def compute_model_overlap(
    model_name: str,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    min_layer = int(config["min_layer"])
    core_effect_mass = float(config["core_effect_mass"])
    prompt_paths = config["prompt_paths"]

    feature_dfs = {
        prompt_name: load_and_preprocess(path, min_layer)
        for prompt_name, path in prompt_paths.items()
    }

    pairwise_rows = []

    for emotion_column in EMOTION_COLUMNS:
        emotion = emotion_column.replace("logit_", "")

        core_sets = {}
        for prompt_name, df in feature_dfs.items():
            effect_dict = get_effect_dictionary(
                df,
                emotion_column,
            )
            core_sets[prompt_name] = get_core_mass_features(
                effect_dict,
                core_effect_mass,
            )

        for prompt_1, prompt_2 in combinations(core_sets.keys(), 2):
            set_1 = core_sets[prompt_1]
            set_2 = core_sets[prompt_2]

            pairwise_rows.append(
                {
                    "model": model_name,
                    "emotion": emotion,
                    "prompt_1": prompt_1,
                    "prompt_2": prompt_2,
                    "core_mass_threshold": core_effect_mass,
                    "num_core_features_1": len(set_1),
                    "num_core_features_2": len(set_2),
                    "core_shared": len(set_1 & set_2),
                    "core_overlap_coefficient": overlap_coefficient(
                        set_1,
                        set_2,
                    ),
                }
            )

    pairwise_df = pd.DataFrame(pairwise_rows)

    summary_df = (
        pairwise_df.groupby(
            ["model", "emotion"],
            as_index=False,
        )
        .agg(
            mean_core_overlap_coefficient=(
                "core_overlap_coefficient",
                "mean",
            ),
            std_core_overlap_coefficient=(
                "core_overlap_coefficient",
                "std",
            ),
            min_core_overlap_coefficient=(
                "core_overlap_coefficient",
                "min",
            ),
            max_core_overlap_coefficient=(
                "core_overlap_coefficient",
                "max",
            ),
            num_prompt_pairs=(
                "core_overlap_coefficient",
                "count",
            ),
        )
    )

    return pairwise_df, summary_df


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    all_pairwise = []
    all_summaries = []

    for model_name, config in MODEL_CONFIGS.items():
        print(f"Processing {model_name}...")

        pairwise_df, summary_df = compute_model_overlap(
            model_name,
            config,
        )

        all_pairwise.append(pairwise_df)
        all_summaries.append(summary_df)

    pairwise_all = pd.concat(
        all_pairwise,
        ignore_index=True,
    )
    summary_all = pd.concat(
        all_summaries,
        ignore_index=True,
    )

    # Row = model, columns = emotions.
    paper_table = (
        summary_all.pivot(
            index="model",
            columns="emotion",
            values="mean_core_overlap_coefficient",
        )
        .reindex(
            index=list(MODEL_CONFIGS.keys()),
            columns=EMOTION_ORDER,
        )
        .reset_index()
        .rename(
            columns={
                "model": "Model",
                "anger": "Anger",
                "disgust": "Disgust",
                "fear": "Fear",
                "joy": "Joy",
                "sadness": "Sadness",
                "surprise": "Surprise",
            }
        )
    )

    pairwise_all.to_csv(
        OUTPUT_DIR / "core_overlap_pairwise_all_models.csv",
        index=False,
    )
    summary_all.to_csv(
        OUTPUT_DIR / "core_overlap_summary_all_models_long.csv",
        index=False,
    )
    paper_table.to_csv(
        OUTPUT_DIR / "core_overlap_coefficient_by_model_emotion.csv",
        index=False,
    )

    latex = paper_table.to_latex(
        index=False,
        float_format=lambda x: f"{x:.3f}",
        column_format="lcccccc",
        escape=False,
    )
    (
        OUTPUT_DIR
        / "core_overlap_coefficient_by_model_emotion.tex"
    ).write_text(latex, encoding="utf-8")

    print("\nMean core overlap coefficient across all six prompt pairs")
    print("=" * 100)
    print(
        paper_table.to_string(
            index=False,
            float_format=lambda x: f"{x:.3f}",
        )
    )

    print(f"\nSaved outputs to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()